In [0]:
from pyspark.sql import functions as F

# Read source file
df = spark.read.option("header", "true").csv("/Workspace/Users/pamkumari33@gmail.com/databricks_sales_capstone/data/sales_source_1500.csv")

print("Total Records :", df.count())

# Quality checks
null_customer = df.filter(F.col("customer_id").isNull()).count()

invalid_quantity = df.filter(
    F.col("quantity").cast("int") <= 0
).count()

invalid_amount = df.filter(
    F.col("net_amount").cast("double") < 0
).count()

invalid_product = df.filter(
    F.upper(F.col("product_id")) == "UNKNOWN"
).count()

report = [
    ("TOTAL_RECORDS", df.count()),
    ("NULL_CUSTOMER", null_customer),
    ("INVALID_QUANTITY", invalid_quantity),
    ("INVALID_AMOUNT", invalid_amount),
    ("INVALID_PRODUCT", invalid_product)
]

quality_df = spark.createDataFrame(report, ["check_name", "record_count"])

display(quality_df)

quality_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.capstone_data_quality_report")